### murdermysterylog 전처리
- 소스: `murdermysterylog.csv`
- 결과: `dasol_murdermysterylog_final.csv`

#### 전처리 항목
1. `play_time` 단위 제거 → 정수형 변환
2. `players` 범위 분리 → `min_players` / `max_players`
3. `players` 컬럼 제거
4. `reviews` 구분자 정규화 (`\n` + `||` 혼용 → `||` 통일)

In [1]:
import pandas as pd
import re

df = pd.read_csv('data/murdermysterylog.csv')

print(f'원본 shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')
df.head(3)

원본 shape: (281, 11)
컬럼: ['url', 'name', 'rating', 'players', 'play_time', 'description', '시리즈', '제작', '출판사', '국내 출판사', 'reviews']


,url,name,rating,players,play_time,description,시리즈,제작,출판사,국내 출판사,reviews
0,https://murdermysterylog.com/detail/0,구두룡 저택의 살인,3.3,7 ~ 9 명,120 분,"저택 지하실에서 영능력자가 시체로 발견되었다! \r\n\r\n어젯밤, 구두룡 저택에...",미스터리 파티 인 더 박스,아키구치 기구루,Group SNE | cosaic,MTS 게임즈,RP를 재밌게 한 머미\n추리 자체는 무난했다 || 역시 이것도 리뷰를 쓰는 지금은...
1,https://murdermysterylog.com/detail/1,몇 번이고 푸른 달에 불을 붙였다,4.2,6 ~ 7 명,150 분,"1962년 11월, 로자넬라의 보스는 자신의 저택에 조직의 핵심 인물들을 초대해 만...",미스터리 파티 인 더 박스,코노 유타카,Group SNE | cosaic,MTS 게임즈,"무난하다고 본다.\n입문작으로도 좋고, 너무 쉬운 편도 아닌 딱 균형잡힌 레벨이 아..."
2,https://murdermysterylog.com/detail/10,문스톤 저택 살인사건,2.9,4 명,90 분,"20세기 중반, 미국 남부. 추리소설을 좋아하는 귀족, 문스톤 경이 희대의 명탐정 ...",머더 미스터리 미니,야스다 히토시,Group SNE | cosaic,언더독 게임즈,"13.(26.05.04)(나, ㅇ현, 2ㅈ원)\n범인X, 검거X\n\n마지막에 범인..."


#### 1. play_time 전처리
- 원본: `"120분"` 형태의 문자열
- 처리: `"분"` 제거 후 정수형(`int`) 변환
- 이유: RAG 필터링 시 수치 비교 가능하도록 정규화
  - ex) "2시간 이내 게임", "플레이 타임 짧은 게임"

In [2]:
# 처리 전 확인
print('처리 전:')
print(df['play_time'].head())

# "분" 제거 후 정수형 변환
df['play_time'] = df['play_time'].str.replace('분', '').str.strip().astype(int)

# 처리 후 확인
print('\n처리 후:')
print(df['play_time'].head())

처리 전:
0    120 분
1    150 분
2     90 분
3    270 분
4    120 분
Name: play_time, dtype: str

처리 후:
0    120
1    150
2     90
3    270
4    120
Name: play_time, dtype: int64


#### 2. players 전처리 → min_players / max_players 분리
- 원본: `"7 ~ 9 명"`, `"4 명"` 형태의 문자열
- 처리: `"명"` 제거 후 `min_players` / `max_players` 두 컬럼으로 분리
- 이유: 문자열 상태에서는 수치 비교 및 필터링 불가
  - ex) "4인 플레이 가능한 게임" → `min_players <= 4 <= max_players` 조건 검색 가능

| 기존 값 | min_players | max_players |
| --- | --- | --- |
| `7 ~ 9 명` | 7 | 9 |
| `4 ~ 5 명` | 4 | 5 |
| `4 명` | 4 | 4 |

In [3]:
# 처리 전 확인
print('처리 전:')
print(df['players'].head(10))

def parse_players(val):
    if pd.isna(val):
        return None, None
    # "명"과 공백 제거
    val = str(val).replace('명', '').replace(' ', '')
    if '~' in val:
        # "7~9" → min=7, max=9
        parts = val.split('~')
        min_p, max_p = int(parts[0]), int(parts[1])
    else:
        # "4" → min=4, max=4
        min_p = max_p = int(val)
    return min_p, max_p

df[['min_players', 'max_players']] = df['players'].apply(
    lambda x: pd.Series(parse_players(x))
)

# 처리 후 확인
print('\n처리 후:')
print(df[['min_players', 'max_players']].head(10))

처리 전:
0    7 ~ 9 명
1    6 ~ 7 명
2        4 명
3    4 ~ 5 명
4        4 명
5        6 명
6    1 ~ 6 명
7    1 ~ 6 명
8    1 ~ 6 명
9        5 명
Name: players, dtype: str

처리 후:
   min_players  max_players
0            7            9
1            6            7
2            4            4
3            4            5
4            4            4
5            6            6
6            1            6
7            1            6
8            1            6
9            5            5


#### 3. players 컬럼 제거
- `min_players` / `max_players`로 대체되었으므로 원본 `players` 컬럼 삭제
- 중복 정보 제거로 데이터 구조 단순화

In [4]:
# players 컬럼 제거
df = df.drop(columns=['players'])

print(f'제거 후 컬럼: {df.columns.tolist()}')

제거 후 컬럼: ['url', 'name', 'rating', 'play_time', 'description', '시리즈', '제작', '출판사', '국내 출판사', 'reviews', 'min_players', 'max_players']


#### 4. reviews 구분자 정규화
- 원본: `\n`과 `||`가 혼용된 형태로 여러 리뷰가 하나의 문자열로 저장
- 처리: 두 구분자를 모두 분리 기준으로 사용 후 `||`로 통일하여 저장
- 이유: 구분자 혼용 시 파싱 일관성이 없어 RAG 청킹 시 오류 발생 가능
  - 통일 후 `split(" || ")`로 리뷰 단위 청킹 가능

In [5]:
def parse_reviews(val):
    if pd.isna(val):
        return ''
    # \n 또는 || 기준으로 분리 후 빈 문자열 제거
    reviews = [r.strip() for r in re.split(r'\n|\|\|', str(val)) if r.strip()]
    # || 구분자로 통일하여 합치기
    return ' || '.join(reviews)

# 처리 전 확인
print('처리 전:')
print(df['reviews'][0][:100])

df['reviews'] = df['reviews'].apply(parse_reviews)

# 처리 후 확인
print('\n처리 후:')
print(df['reviews'][0][:100])

처리 전:
RP를 재밌게 한 머미
추리 자체는 무난했다 || 역시 이것도 리뷰를 쓰는 지금은 이것저것 짬뽕돼서 기억이 애매하다. 다만 이것 역시 파티시리즈의 기분과 분위기는 잘 녹아있는 것이

처리 후:
RP를 재밌게 한 머미 || 추리 자체는 무난했다 || 역시 이것도 리뷰를 쓰는 지금은 이것저것 짬뽕돼서 기억이 애매하다. 다만 이것 역시 파티시리즈의 기분과 분위기는 잘 녹아있는


#### 5. 최종 저장

In [6]:
df.to_csv('data/murdermysterylog_final.csv', index=False, encoding='utf-8-sig')

print(f'저장 완료!')
print(f'shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')
df.head(3)

저장 완료!
shape: (281, 12)
컬럼: ['url', 'name', 'rating', 'play_time', 'description', '시리즈', '제작', '출판사', '국내 출판사', 'reviews', 'min_players', 'max_players']


,url,name,rating,play_time,description,시리즈,제작,출판사,국내 출판사,reviews,min_players,max_players
0,https://murdermysterylog.com/detail/0,구두룡 저택의 살인,3.3,120,"저택 지하실에서 영능력자가 시체로 발견되었다! \r\n\r\n어젯밤, 구두룡 저택에...",미스터리 파티 인 더 박스,아키구치 기구루,Group SNE | cosaic,MTS 게임즈,RP를 재밌게 한 머미 || 추리 자체는 무난했다 || 역시 이것도 리뷰를 쓰는 지...,7,9
1,https://murdermysterylog.com/detail/1,몇 번이고 푸른 달에 불을 붙였다,4.2,150,"1962년 11월, 로자넬라의 보스는 자신의 저택에 조직의 핵심 인물들을 초대해 만...",미스터리 파티 인 더 박스,코노 유타카,Group SNE | cosaic,MTS 게임즈,"무난하다고 본다. || 입문작으로도 좋고, 너무 쉬운 편도 아닌 딱 균형잡힌 레벨이...",6,7
2,https://murdermysterylog.com/detail/10,문스톤 저택 살인사건,2.9,90,"20세기 중반, 미국 남부. 추리소설을 좋아하는 귀족, 문스톤 경이 희대의 명탐정 ...",머더 미스터리 미니,야스다 히토시,Group SNE | cosaic,언더독 게임즈,"13.(26.05.04)(나, ㅇ현, 2ㅈ원) || 범인X, 검거X || 마지막에 ...",4,4


In [7]:
import pandas as pd

df = pd.read_csv("data/murdermysterylog_final.csv")

# 1. 기본 정보
print(f"shape: {df.shape}")
print(f"컬럼: {df.columns.tolist()}")

# 2. 데이터 타입 확인
print(f"\n데이터 타입:")
print(df.dtypes)

# 3. 결측값 확인
print(f"\n결측값:")
print(df.isna().sum())

# 4. 각 전처리 항목 샘플 확인
print(f"\nplay_time 샘플: {df['play_time'].head().tolist()}")
print(f"min_players 샘플: {df['min_players'].head().tolist()}")
print(f"max_players 샘플: {df['max_players'].head().tolist()}")
print(f"\nreviews 샘플:")
print(df['reviews'][0][:200])

shape: (281, 12)
컬럼: ['url', 'name', 'rating', 'play_time', 'description', '시리즈', '제작', '출판사', '국내 출판사', 'reviews', 'min_players', 'max_players']

데이터 타입:
url                str
name               str
rating         float64
play_time        int64
description        str
시리즈                str
제작                 str
출판사                str
국내 출판사             str
reviews            str
min_players      int64
max_players      int64
dtype: object

결측값:
url              0
name             0
rating          51
play_time        0
description      9
시리즈            198
제작             155
출판사            175
국내 출판사         114
reviews          8
min_players      0
max_players      0
dtype: int64

play_time 샘플: [120, 150, 90, 270, 120]
min_players 샘플: [7, 6, 4, 4, 4]
max_players 샘플: [9, 7, 4, 5, 4]

reviews 샘플:
RP를 재밌게 한 머미 || 추리 자체는 무난했다 || 역시 이것도 리뷰를 쓰는 지금은 이것저것 짬뽕돼서 기억이 애매하다. 다만 이것 역시 파티시리즈의 기분과 분위기는 잘 녹아있는 것이 좋다. || 밀담 O || 파티 시리즈를 구두룡으로 입문했는데 매우 재밌었다. || 초창기에 접했던 가장 재미있게 했던 머미 || 7人 || 파티시리즈 입문